# Survival Retention Analysis

## Import & Check Data

1. load

In [2]:
import pandas as pd
import numpy as np

DATA = "../data/"  # change to your folder
subs = pd.read_csv(DATA + "subscribers.csv")
events = pd.read_csv(DATA + "subscription_events.csv")

print(subs.shape, events.shape)
display(subs.head())
display(events.head())

(59940, 9) (244066, 6)


,subscriber_id,signup_ts,platform,plan,acquisition_channel,country,promo_code,first_week_sessions,completed_onboarding
0,sub_94070706,2025-02-18 20:03:32,iOS,monthly,app_store_ads,US,NaN,5,0
1,sub_22238885,2025-08-14 17:53:40,iOS,annual,paid_social,US,NaN,6,1
2,sub_50394643,2024-05-15 22:06:40,iOS,annual,organic,US,NaN,8,0
3,sub_17974224,2025-04-08 01:25:57,iOS,annual,organic,US,NaN,0,0
4,sub_19734259,2026-02-02 18:06:29,iOS,monthly,paid_social,US,NaN,5,1


,event_id,subscriber_id,event_ts,event_type,amount_usd,event_detail
0,evt_0000001,sub_64689158,2024-01-01 00:15:50,trial_start,0.0,NaN
1,evt_0000002,sub_30898631,2024-01-01 00:20:29,trial_start,0.0,NaN
2,evt_0000003,sub_49672362,2024-01-01 00:24:34,trial_start,0.0,NaN
3,evt_0000004,sub_13266245,2024-01-01 00:50:37,trial_start,0.0,NaN
4,evt_0000005,sub_49672362,2024-01-01 01:11:21,autorenew_off,0.0,trial


2. types and missing values

In [3]:
print(subs.dtypes, "\n")
print(events.dtypes, "\n")

print("subs nulls:\n", subs.isna().sum(), "\n")
print("events nulls:\n", events.isna().sum())

subscriber_id             str
signup_ts                 str
platform                  str
plan                      str
acquisition_channel       str
country                   str
promo_code                str
first_week_sessions     int64
completed_onboarding    int64
dtype: object 

event_id             str
subscriber_id        str
event_ts             str
event_type           str
amount_usd       float64
event_detail         str
dtype: object 

subs nulls:
 subscriber_id               0
signup_ts                   0
platform                    0
plan                        0
acquisition_channel      3019
country                   873
promo_code              57129
first_week_sessions         0
completed_onboarding        0
dtype: int64 

events nulls:
 event_id              0
subscriber_id         0
event_ts              0
event_type            0
amount_usd          260
event_detail     149948
dtype: int64


3. categorical values & keys

In [4]:
for col in ["platform", "plan", "acquisition_channel", "country", "promo_code"]:
    print(subs[col].value_counts(dropna=False), "\n")

print(events["event_type"].value_counts(), "\n")
print(events["event_detail"].value_counts(dropna=False))

print("\nduplicate subscriber_id:", subs["subscriber_id"].duplicated().sum())
print("duplicate event_id:", events["event_id"].duplicated().sum())
print("exact duplicate event rows:", events.duplicated().sum())

platform
iOS        31822
Android    17214
Web         8577
ios          651
IOS          631
ANDROID      361
android      351
web          167
WEB          166
Name: count, dtype: int64 

plan
annual     34364
monthly    25576
Name: count, dtype: int64 

acquisition_channel
organic          16818
paid_social      11036
app_store_ads     8521
paid_search       7138
influencer        6688
referral          5543
NaN               3019
Organic            355
Paid_social        228
App_store_ads      184
Paid_search        154
Influencer         145
Referral           111
Name: count, dtype: int64 

country
US       36629
Other     8903
UK        5319
CA        3523
DE        2362
AU        2331
NaN        873
Name: count, dtype: int64 

promo_code
NaN      57129
NY_20     1888
BF_40      923
Name: count, dtype: int64 

event_type
trial_start          59959
renewal              55059
expired              51033
autorenew_off        47076
trial_converted      19229
payment_failed        532

4. parse dates

In [5]:
subs["signup_ts_p"] = pd.to_datetime(subs["signup_ts"], errors="coerce")
events["event_ts_p"] = pd.to_datetime(events["event_ts"], errors="coerce")

print("signup_ts failed to parse:", subs["signup_ts_p"].isna().sum())
print("event_ts failed to parse:", events["event_ts_p"].isna().sum())
print(subs.loc[subs["signup_ts_p"].isna(), "signup_ts"].head())

print(subs["signup_ts_p"].min(), subs["signup_ts_p"].max())
print(events["event_ts_p"].min(), events["event_ts_p"].max())

signup_ts failed to parse: 248
event_ts failed to parse: 0
492     03/06/2026
531     07/24/2026
693     05/29/2024
821     03/29/2024
1116    08/21/2024
Name: signup_ts, dtype: str
2024-01-01 00:15:50 2026-09-30 23:54:27
2024-01-01 00:15:50 2026-10-31 23:36:20


5. do the two tables agree?

In [6]:
sub_ids, ev_ids = set(subs["subscriber_id"]), set(events["subscriber_id"])
print("in events, not in subscribers:", len(ev_ids - sub_ids))
print("in subscribers, no events:", len(sub_ids - ev_ids))

# every subscriber should have exactly one trial_start
ts = events[events.event_type == "trial_start"].groupby("subscriber_id").size()
print("subscribers missing a trial_start:", len(sub_ids - set(ts.index)))
print("with more than one:", (ts > 1).sum())

# events happening before the trial started
first = events.merge(subs[["subscriber_id", "signup_ts_p"]], on="subscriber_id")
print("events before signup_ts:", (first["event_ts_p"] < first["signup_ts_p"]).sum())

in events, not in subscribers: 120
in subscribers, no events: 0
subscribers missing a trial_start: 252
with more than one: 152
events before signup_ts: 539


6. money & monthly volume

In [7]:
print(events.groupby("event_type")["amount_usd"].describe())
print(events.groupby(["event_type", "amount_usd"]).size().loc[["trial_converted", "renewal"]].head(15))

monthly = events.set_index("event_ts_p").groupby([pd.Grouper(freq="MS"), "event_type"]).size().unstack(fill_value=0)
display(monthly.tail(12))

                     count       mean        std    min    25%    50%    75%  \
event_type                                                                     
autorenew_off      47076.0   0.000000   0.000000   0.00   0.00   0.00   0.00   
autorenew_on         742.0   0.000000   0.000000   0.00   0.00   0.00   0.00   
expired            51033.0   0.000000   0.000000   0.00   0.00   0.00   0.00   
payment_failed      5327.0   0.000000   0.000000   0.00   0.00   0.00   0.00   
payment_recovered   2676.0  29.188804  26.694073   9.99   9.99  12.99  69.99   
reactivated         2604.0  32.646836  25.612200   5.99   9.99  12.99  69.99   
refund               361.0 -51.809945  26.274475 -69.99 -69.99 -69.99 -12.99   
renewal            54799.0  15.650040  15.836897   0.00   9.99   9.99  12.99   
trial_converted    19229.0  41.681435  28.647974   0.00  12.99  55.99  69.99   
trial_start        59959.0   0.000000   0.000000   0.00   0.00   0.00   0.00   

                     max  
event_type  

event_type,autorenew_off,autorenew_on,expired,payment_failed,payment_recovered,reactivated,refund,renewal,trial_converted,trial_start
event_ts_p,,,,,,,,,,
2025-11-01,1665,31,1647,194,102,86,15,2062,661,2393
2025-12-01,1907,32,2249,198,104,111,13,2290,897,2255
2026-01-01,2604,31,2789,298,135,112,25,2485,1092,3791
2026-02-01,1717,29,2034,207,135,113,15,2398,752,1891
2026-03-01,1803,43,1969,269,129,107,12,2673,754,2257
2026-04-01,1794,32,1937,241,133,112,14,2694,724,2145
2026-05-01,1876,26,2064,238,112,113,13,2846,743,2272
2026-06-01,1662,35,1823,217,117,118,17,2788,674,1922
2026-07-01,1714,37,1827,265,140,105,15,2937,694,1979


## Import & Check Data